<a href="https://colab.research.google.com/github/juliasharmankina-web/Exploratory-data-analysis-for-online-store/blob/main/A_B_Test_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import auth
from google.cloud import bigquery
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import plotly.express as px
import scipy.stats as stats

auth.authenticate_user()

client = bigquery.Client(project="data-analytics-mate")

##SQL Query

In [ ]:
query="""
WITH session_info AS (
  SELECT
    s.date,
    s.ga_session_id,
    sp.country,
    sp.device,
    sp.continent,
    sp.channel,
    ab.test,
    ab.test_group
  FROM `DA.ab_test` ab
  JOIN `DA.session` s
    ON ab.ga_session_id = s.ga_session_id
  JOIN `DA.session_params` sp
    ON sp.ga_session_id = ab.ga_session_id
),
session_with_orders AS (
  SELECT
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    COUNT(DISTINCT o.ga_session_id) AS session_with_orders
  FROM `DA.order` o
  JOIN session_info
    ON o.ga_session_id = session_info.ga_session_id
  GROUP BY
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group
),
events AS (
  SELECT
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    sp.event_name,
    COUNT(sp.ga_session_id) AS event_cnt
  FROM `DA.event_params` sp
  JOIN session_info
    ON sp.ga_session_id = session_info.ga_session_id
  GROUP BY
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    sp.event_name
),
session AS (
  SELECT
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    COUNT(DISTINCT session_info.ga_session_id) AS session_cnt
  FROM session_info
  GROUP BY
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group
),
account AS (
  SELECT
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    COUNT(DISTINCT acs.ga_session_id) AS new_account_cnt
  FROM `DA.account_session` acs
  JOIN session_info
    ON acs.ga_session_id = session_info.ga_session_id
  GROUP BY
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group
)
SELECT
  session_with_orders.date,
  session_with_orders.country,
  session_with_orders.device,
  session_with_orders.continent,
  session_with_orders.channel,
  session_with_orders.test,
  session_with_orders.test_group,
  'session with orders' AS event_name,
  session_with_orders.session_with_orders AS value
FROM session_with_orders
UNION ALL
SELECT
  events.date,
  events.country,
  events.device,
  events.continent,
  events.channel,
  events.test,
  events.test_group,
  events.event_name,
  events.event_cnt AS value
FROM events
UNION ALL
SELECT
  session.date,
  session.country,
  session.device,
  session.continent,
  session.channel,
  session.test,
  session.test_group,
  'session' AS event_name,
  session.session_cnt AS value
FROM session
UNION ALL
SELECT
  account.date,
  account.country,
  account.device,
  account.continent,
  account.channel,
  account.test,
  account.test_group,
  'new account' AS event_name,
  account.new_account_cnt AS value
FROM account;
"""
df_test = client.query(query).result().to_dataframe(create_bqstorage_client=False)
df_test.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-12-08,Palestine,desktop,Asia,Direct,4,2,new account,1
1,2020-12-08,Palestine,desktop,Asia,Direct,3,2,new account,1
2,2020-11-06,Puerto Rico,desktop,Americas,Social Search,2,2,new account,1
3,2020-11-06,Puerto Rico,desktop,Americas,Social Search,1,1,new account,1
4,2020-12-08,Croatia,desktop,Europe,Direct,4,2,new account,1


In [ ]:
print(df_test.shape)
print("")
print(df_test.info())

(800996, 9)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        800996 non-null  dbdate
 1   country     800996 non-null  object
 2   device      800996 non-null  object
 3   continent   800996 non-null  object
 4   channel     800996 non-null  object
 5   test        800996 non-null  Int64 
 6   test_group  800996 non-null  Int64 
 7   event_name  800996 non-null  object
 8   value       800996 non-null  Int64 
dtypes: Int64(3), dbdate(1), object(5)
memory usage: 57.3+ MB
None


##Calculating the statistical significance for following metrics

In [ ]:
#checking event names
df_test.event_name.unique()

array(['new account', 'session with orders', 'user_engagement',
       'session_start', 'page_view', 'first_visit', 'scroll', 'view_item',
       'view_promotion', 'add_shipping_info', 'add_to_cart',
       'view_search_results', 'select_promotion', 'begin_checkout',
       'select_item', 'add_payment_info', 'click', 'session',
       'view_item_list'], dtype=object)

In [ ]:
import statsmodels.api as sm
import pandas as pd
from IPython.display import display
import os
from google.colab import drive # Import drive

# Define the metrics to analyze (using the exact event_name from df_test.event_name.unique())
metric_event_names = [
    'add_payment_info',
    'add_shipping_info',
    'begin_checkout',
    'new account'
]

# Define the columns for breakdown analysis
breakdown_columns = ['channel', 'device', 'country', 'continent']

# Ensure 'test_group' is numeric. Use errors='coerce' to handle non-numeric gracefully
# and then drop any rows where conversion failed, though it should already be numeric from previous cells.
df_test['test_group'] = pd.to_numeric(df_test['test_group'], errors='coerce')
df_test.dropna(subset=['test_group'], inplace=True)

# Initialize a list to store all results
all_test_results = []

# --- Function to perform A/B test analysis for a given segment ---
def perform_ab_test_for_segment(df_segment, metric_event_name, breakdown_dimension, breakdown_value):
    """
    Performs A/B test analysis (Z-test for proportions) for a specific metric
    within a given breakdown dimension and value.

    Args:
        df_segment (pd.DataFrame): The DataFrame filtered for a specific breakdown value.
        metric_event_name (str): The event name for which to calculate the conversion.
        breakdown_dimension (str): The column used for breakdown (e.g., 'channel').
        breakdown_value (str): The specific value within the breakdown_dimension being analyzed.

    Returns:
        dict: A dictionary containing the A/B test results for the segment.
    """

    sessions_group_1 = df_segment[(df_segment.test_group == 1) & (df_segment.event_name == 'session')]['value'].sum()
    sessions_group_2 = df_segment[(df_segment.test_group == 2) & (df_segment.event_name == 'session')]['value'].sum()

    purchases_group_1 = df_segment[(df_segment.test_group == 1) & (df_segment.event_name == metric_event_name)]['value'].sum()
    purchases_group_2 = df_segment[(df_segment.test_group == 2) & (df_segment.event_name == metric_event_name)]['value'].sum()

    cr_group_1 = (purchases_group_1 / sessions_group_1) * 100 if sessions_group_1 > 0 else 0.0
    cr_group_2 = (purchases_group_2 / sessions_group_2) * 100 if sessions_group_2 > 0 else 0.0

    metric_change_pct = 0.0
    if cr_group_2 > 0:
        metric_change_pct = ((cr_group_1 - cr_group_2) / cr_group_2) * 100
    elif cr_group_1 > 0: # Control conversion is 0 but test is > 0, indicating infinite lift
        metric_change_pct = float('inf')
    # If both are 0, metric_change_pct remains 0.0

    z_stat, p_value, significant_result = None, None, 'N/A'

    # Only perform z-test if there are sessions in both groups and at least one purchase total
    if sessions_group_1 > 0 and sessions_group_2 > 0:
        try:
            if purchases_group_1 + purchases_group_2 > 0: # At least one success for the test to be meaningful
                z_stat, p_value = sm.stats.proportions_ztest(
                    [purchases_group_1, purchases_group_2],
                    [sessions_group_1, sessions_group_2]
                )
                significant_result = 'TRUE' if p_value < 0.05 else 'FALSE'
            else: # No purchases in either group, so no statistical difference
                significant_result = 'FALSE (no purchases)'
        except ValueError: # Catch cases where z-test might fail (e.g., counts or nobs issues)
            significant_result = 'Error in Z-test calculation'
    else:
        significant_result = 'N/A (no sessions in one or both groups)'

    return {
        'breakdown_dimension': breakdown_dimension,
        'breakdown_value': breakdown_value,
        'metric_name': metric_event_name,
        'numerator_control': purchases_group_2,
        'denominator_control': sessions_group_2,
        'conversion_rate_control': round(cr_group_2, 4),
        'numerator_test': purchases_group_1,
        'denominator_test': sessions_group_1,
        'conversion_rate_test': round(cr_group_1, 4),
        'metric_change_pct': round(metric_change_pct, 2) if pd.notna(metric_change_pct) and metric_change_pct != float('inf') else metric_change_pct,
        'z_stat': round(z_stat, 4) if z_stat is not None else None,
        'p_value': round(p_value, 4) if p_value is not None else None,
        'significant': significant_result
    }

# --- Initial overall analysis (representing the original cell's intent) ---
# This provides a summary without further breakdown
for metric in metric_event_names:
    overall_result = perform_ab_test_for_segment(
        df_test, # Pass the full df_test for overall calculation
        metric,
        'Overall', # Dimension name for overall results
        'All' # Value for overall results
    )
    all_test_results.append(overall_result)

# --- Perform breakdown analysis for each specified column and metric ---
for col in breakdown_columns:
    unique_values = df_test[col].unique()
    for value in unique_values:
        # Filter for the specific breakdown value and create a copy to avoid SettingWithCopyWarning
        df_segment = df_test[df_test[col] == value].copy()
        for metric in metric_event_names:
            result = perform_ab_test_for_segment(df_segment, metric, col, value)
            all_test_results.append(result)

# Convert all results to a DataFrame
df_all_results = pd.DataFrame(all_test_results)

# --- Filter for display based on user request: "only for the main metrics, remove all others" ---
# This means displaying only the 'Overall' results for the defined main metrics.
df_main_metrics_summary = df_all_results[
    (df_all_results['breakdown_dimension'] == 'Overall') &
    (df_all_results['metric_name'].isin(metric_event_names))
].copy()

# Rename and reorder columns for a clearer presentation in this summary
df_main_metrics_summary = df_main_metrics_summary.rename(columns={
    'metric_name': 'Metric',
    'conversion_rate_control': 'CR Control (%)',
    'conversion_rate_test': 'CR Test (%)',
    'metric_change_pct': 'Lift (%)',
    'p_value': 'P-value',
    'significant': 'Significant'
})

df_main_metrics_summary = df_main_metrics_summary[[
    'Metric',
    'CR Control (%)',
    'CR Test (%)',
    'Lift (%)',
    'P-value',
    'Significant'
]]

# Display only the summary for main metrics
display(df_main_metrics_summary)

# --- Save df_all_results to Google Drive in CSV format ---
# Ensure Google Drive is mounted. Skip if already mounted to avoid errors.
# drive.mount('/content/drive') # Commented out, assuming drive is already mounted.

directory_path = '/content/drive/MyDrive/mate_academy'
file_path = os.path.join(directory_path, 'ab_test_all_results.csv')

# Create directory if it doesn't exist
if not os.path.exists(directory_path):
    os.makedirs(directory_path)
    print(f"Directory '{directory_path}' created.")

# Save the DataFrame to CSV
df_all_results.to_csv(file_path, index=False)
print(f"DataFrame saved to '{file_path}'")

,Metric,CR Control (%),CR Test (%),Lift (%),P-value,Significant
0,add_payment_info,4.4042,4.3102,-2.13,0.0902,FALSE
1,add_shipping_info,6.2265,6.2480,0.35,0.7435,FALSE
2,begin_checkout,11.0196,11.1141,0.86,0.2677,FALSE
3,new account,8.2556,8.4197,1.99,0.0288,TRUE


DataFrame saved to '/content/drive/MyDrive/mate_academy/ab_test_all_results.csv'


## Breakdown by Channel

In [ ]:
# Filter df_all_results for 'channel' breakdown
df_channel_summary = df_all_results[
    (df_all_results['breakdown_dimension'] == 'channel') &
    (df_all_results['metric_name'].isin(summary_metric_names)) # Using the same metrics as the overall summary
].copy()

# Rename and reorder columns for a clearer presentation
df_channel_summary = df_channel_summary.rename(columns={
    'breakdown_value': 'Channel',
    'metric_name': 'Metric',
    'conversion_rate_control': 'CR Control (%)',
    'conversion_rate_test': 'CR Test (%)',
    'metric_change_pct': 'Lift (%)',
    'p_value': 'P-value',
    'significant': 'Significant'
})

df_channel_summary = df_channel_summary[[
    'Channel',
    'Metric',
    'CR Control (%)',
    'CR Test (%)',
    'Lift (%)',
    'P-value',
    'Significant'
]]

# Display the DataFrame
display(df_channel_summary)

,Channel,Metric,CR Control (%),CR Test (%),Lift (%),P-value,Significant
4,Direct,add_payment_info,4.1977,4.2092,0.27,0.9195,FALSE
5,Direct,add_shipping_info,5.9784,6.1335,2.59,0.2481,FALSE
6,Direct,begin_checkout,10.7475,10.8028,0.51,0.7515,FALSE
7,Direct,new account,8.2082,8.4689,3.18,0.0939,FALSE
8,Social Search,add_payment_info,6.2342,6.3289,1.52,0.6819,FALSE
9,Social Search,add_shipping_info,8.2805,8.3626,0.99,0.7552,FALSE
10,Social Search,begin_checkout,14.8452,15.0654,1.48,0.5170,FALSE
11,Social Search,new account,8.0904,8.2988,2.58,0.4254,FALSE
12,Organic Search,add_payment_info,3.4103,3.6195,6.13,0.0129,TRUE
13,Organic Search,add_shipping_info,5.1717,5.5452,7.22,0.0003,TRUE


In [ ]:
import os

# Define the directory path and file name for the channel summary
directory_path = '/content/drive/MyDrive/mate_academy'
channel_summary_file_name = 'ab_test_channel_summary.csv'
channel_summary_file_path = os.path.join(directory_path, channel_summary_file_name)

# Create directory if it doesn't exist (exist_ok=True prevents error if it already exists)
if not os.path.exists(directory_path):
    os.makedirs(directory_path, exist_ok=True)
    print(f"Directory '{directory_path}' created.")

# Save the df_channel_summary DataFrame to CSV
df_channel_summary.to_csv(channel_summary_file_path, index=False)
print(f"DataFrame 'df_channel_summary' saved to '{channel_summary_file_path}'")

DataFrame 'df_channel_summary' saved to '/content/drive/MyDrive/mate_academy/ab_test_channel_summary.csv'


## Breakdown by Device

In [ ]:
# Filter df_all_results for 'device' breakdown
df_device_summary = df_all_results[
    (df_all_results['breakdown_dimension'] == 'device') &
    (df_all_results['metric_name'].isin(summary_metric_names))
].copy()

# Rename and reorder columns for a clearer presentation
df_device_summary = df_device_summary.rename(columns={
    'breakdown_value': 'Device',
    'metric_name': 'Metric',
    'conversion_rate_control': 'CR Control (%)',
    'conversion_rate_test': 'CR Test (%)',
    'metric_change_pct': 'Lift (%)',
    'p_value': 'P-value',
    'significant': 'Significant'
})

df_device_summary = df_device_summary[[
    'Device',
    'Metric',
    'CR Control (%)',
    'CR Test (%)',
    'Lift (%)',
    'P-value',
    'Significant'
]]

# Display the DataFrame
display(df_device_summary)

,Device,Metric,CR Control (%),CR Test (%),Lift (%),P-value,Significant
24,desktop,add_payment_info,4.3296,4.2522,-1.79,0.2821,FALSE
25,desktop,add_shipping_info,6.2524,6.2193,-0.53,0.7005,FALSE
26,desktop,begin_checkout,11.0804,11.0909,0.09,0.9255,FALSE
27,desktop,new account,8.2218,8.4621,2.92,0.0144,TRUE
28,mobile,add_payment_info,4.5789,4.3780,-4.39,0.0250,TRUE
29,mobile,add_shipping_info,6.2696,6.2948,0.40,0.8108,FALSE
30,mobile,begin_checkout,11.0903,11.1534,0.57,0.6437,FALSE
31,mobile,new account,8.2810,8.3354,0.66,0.6493,FALSE
32,tablet,add_payment_info,3.2893,4.6373,40.98,0.0001,TRUE
33,tablet,add_shipping_info,4.7934,6.1775,28.88,0.0008,TRUE


## Breakdown by Continent

In [ ]:
# Filter df_all_results for 'continent' breakdown
df_continent_summary = df_all_results[
    (df_all_results['breakdown_dimension'] == 'continent') &
    (df_all_results['metric_name'].isin(summary_metric_names))
].copy()

# Rename and reorder columns for a clearer presentation
df_continent_summary = df_continent_summary.rename(columns={
    'breakdown_value': 'Continent',
    'metric_name': 'Metric',
    'conversion_rate_control': 'CR Control (%)',
    'conversion_rate_test': 'CR Test (%)',
    'metric_change_pct': 'Lift (%)',
    'p_value': 'P-value',
    'significant': 'Significant'
})

df_continent_summary = df_continent_summary[[
    'Continent',
    'Metric',
    'CR Control (%)',
    'CR Test (%)',
    'Lift (%)',
    'P-value',
    'Significant'
]]

# Display the DataFrame
display(df_continent_summary)

,Continent,Metric,CR Control (%),CR Test (%),Lift (%),P-value,Significant
468,Asia,add_payment_info,4.3954,4.3261,-1.58,0.5426,FALSE
469,Asia,add_shipping_info,6.1378,6.0394,-1.60,0.4601,FALSE
470,Asia,begin_checkout,10.7832,10.5682,-1.99,0.2113,FALSE
471,Asia,new account,8.3232,8.5276,2.46,0.1865,FALSE
472,Americas,add_payment_info,4.4075,4.2227,-4.19,0.0129,TRUE
473,Americas,add_shipping_info,6.2234,6.2547,0.50,0.7237,FALSE
474,Americas,begin_checkout,11.0601,11.1383,0.71,0.4963,FALSE
475,Americas,new account,8.2379,8.4668,2.78,0.0236,TRUE
476,Europe,add_payment_info,4.3889,4.3892,0.01,0.9983,FALSE
477,Europe,add_shipping_info,6.2529,6.3469,1.50,0.5379,FALSE
